# What sets the price of an Athens Airbnb?

Goal: predict the nightly asking price of Athens listings and measure the **Acropolis premium**, meaning how much price changes with distance from the Acropolis, split by room type.

Important: these are asking prices set by hosts, not prices guests paid. Keep that in mind when reading any result.

Plan: load, clean, features, host-grouped split, baseline, Random Forest, LightGBM, evaluate, core finding, possibly underpriced listings.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error
from sklearn.inspection import permutation_importance
from lightgbm import LGBMRegressor

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.features import clean_price, add_features

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

## 1. Load the data
Inside Airbnb detailed listings for Athens, scraped 29 June 2026. Prices are in EUR. Download instructions are in `data/README.md`.

In [ ]:
raw = pd.read_csv(ROOT / "data" / "listings.csv")
print(raw.shape)
raw.head()

In [ ]:
raw.info()

## 2. Clean
Every dropped row gets counted. Those numbers go in the README limitations section.

In [ ]:
df = raw.copy()
df["price"] = clean_price(df["price"])
n_start = len(df)

missing_price = df["price"].isna().sum()
df = df.dropna(subset=["price"])

# Remove placeholder and extreme prices. Check these cutoffs against the histogram below.
low, high = 20, df["price"].quantile(0.99)
outliers = ((df["price"] < low) | (df["price"] > high)).sum()
df = df[(df["price"] >= low) & (df["price"] <= high)]

print(f"Start: {n_start} listings")
print(f"Dropped, missing price: {missing_price}")
print(f"Dropped, price outside {low} to {high:.0f}: {outliers}")
print(f"Remaining: {len(df)}")

In [ ]:
df["price"].hist(bins=60)
plt.xlabel("Nightly asking price (EUR)")
plt.ylabel("Listings")
plt.title("Price distribution after cleaning")
plt.show()

## 3. Features

Size (guests, bedrooms, bathrooms), location (distance to the Acropolis and Syntagma, neighbourhood), quality (review scores, superhost) and host behaviour.

`instant_bookable` is blank for every listing in this scrape, so it is not used.

Left out on purpose: `estimated_revenue_l365d` and the `price_quote_*` columns. They are calculated from price, so using them would leak the answer into the model. `add_features` drops them.

In [ ]:
df = add_features(df)

NUMERIC = ["km_acropolis", "km_syntagma", "accommodates", "bedrooms", "beds", "bathrooms",
           "minimum_nights", "number_of_reviews", "reviews_per_month", "number_of_reviews_ltm",
           "review_scores_rating", "review_scores_location", "host_is_superhost",
           "calculated_host_listings_count", "availability_365",
           "is_multi_host"]
CATEGORICAL = ["room_type", "neighbourhood"]
NUMERIC = [c for c in NUMERIC if c in df.columns]

X = df[NUMERIC + CATEGORICAL].copy()
for c in CATEGORICAL:
    X[c] = X[c].astype("category")
y = np.log1p(df["price"])   # log target: prices are skewed, errors become relative
groups = df["host_id"]
X.describe(include="all").T

## 4. Split by host, not by row

Hosts with many listings often copy the same price across them. A random split would put sibling listings in both train and test, and the model would look better than it is. Grouping by `host_id` keeps each host on one side.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr, te = next(gss.split(X, y, groups))
X_tr, X_te, y_tr, y_te = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
price_te = np.expm1(y_te)
print(len(X_tr), "train /", len(X_te), "test")

## 5. Baseline: median price for the same neighbourhood and room type
Any model has to beat this to be worth posting.

In [ ]:
train_df = X_tr.assign(price=np.expm1(y_tr))
lookup = train_df.groupby(["neighbourhood", "room_type"], observed=True)["price"].median()
fallback = train_df.groupby("room_type", observed=True)["price"].median()

def baseline_predict(frame):
    keys = list(zip(frame["neighbourhood"], frame["room_type"]))
    preds = [lookup.get(k, fallback.get(k[1], train_df["price"].median())) for k in keys]
    return np.array(preds, dtype=float)

results = {"Baseline (median)": mean_absolute_error(price_te, baseline_predict(X_te))}
results

## 6. Models

In [ ]:
rf = Pipeline([
    ("prep", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("num", SimpleImputer(strategy="median"), NUMERIC),
    ])),
    ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                    n_jobs=-1, random_state=SEED)),
])
rf.fit(X_tr, y_tr)
results["Random Forest"] = mean_absolute_error(price_te, np.expm1(rf.predict(X_te)))

lgbm = LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=31,
                     min_child_samples=20, random_state=SEED, verbose=-1)
lgbm.fit(X_tr, y_tr)
results["LightGBM"] = mean_absolute_error(price_te, np.expm1(lgbm.predict(X_te)))

pd.Series(results, name="MAE (EUR)").round(2).sort_values()

## 7. Where is the model wrong? Error by room type

In [ ]:
pred_te = np.expm1(lgbm.predict(X_te))
err = pd.DataFrame({"room_type": X_te["room_type"].values,
                    "abs_err": np.abs(pred_te - price_te.values)})
err.groupby("room_type", observed=True)["abs_err"].agg(["count", "mean"]).round(2)

In [ ]:
imp = permutation_importance(lgbm, X_te, y_te, n_repeats=5, random_state=SEED)
pd.Series(imp.importances_mean, index=X_te.columns).sort_values().plot.barh()
plt.title("Permutation importance (LightGBM, test set)")
plt.tight_layout()
plt.show()

## 8. Core finding: the Acropolis premium

Median asking price by distance band, one line per room type. This chart is the LinkedIn post.

In [ ]:
bands = [0, 0.5, 1, 1.5, 2, 3, 5, 10, np.inf]
labels = ["<0.5", "0.5-1", "1-1.5", "1.5-2", "2-3", "3-5", "5-10", "10+"]
df["km_band"] = pd.cut(df["km_acropolis"], bins=bands, labels=labels)

main_types = df["room_type"].value_counts().index[:2]   # usually entire home and private room
table = (df[df["room_type"].isin(main_types)]
         .groupby(["km_band", "room_type"], observed=True)["price"]
         .median().unstack())

ax = table.plot(marker="o", figsize=(8, 4.5))
ax.set_xlabel("Distance from the Acropolis (km)")
ax.set_ylabel("Median nightly asking price (EUR)")
ax.set_title("The Acropolis premium in Athens Airbnb prices")
plt.tight_layout()
plt.savefig(FIG_DIR / "acropolis_premium.png", dpi=200)
plt.show()
table.round(0)

## 9. Possibly underpriced listings

Out-of-fold predictions (each listing is predicted by a model that never saw its host). Listings asking well below the model are *possibly* underpriced compared with similar listings. This is not proof of lost revenue.

Only entire homes are checked. Shared and hostel rooms are priced per bed, which the model cannot tell apart from a whole apartment, so they produce false alarms.

In [ ]:
oof = np.expm1(cross_val_predict(
    LGBMRegressor(n_estimators=600, learning_rate=0.03, random_state=SEED, verbose=-1),
    X, y, groups=groups, cv=GroupKFold(n_splits=5)))

df["predicted"] = oof.round(0)
df["gap_pct"] = ((df["predicted"] - df["price"]) / df["price"] * 100).round(0)

cols = ["id", "neighbourhood", "room_type", "price", "predicted", "gap_pct", "number_of_reviews"]
homes = df[df["room_type"] == "Entire home/apt"]
under = homes[(homes["gap_pct"] > 40) & (homes["number_of_reviews"] >= 10)]
print(f"{len(under)} of {len(homes)} entire homes ({len(under)/len(homes):.1%}) ask 40%+ below the model and have 10+ reviews")
under.sort_values("gap_pct", ascending=False)[cols].head(15)

## 10. Notes for the README

Fill these in from the outputs above:

- Scrape date of the data:
- Listings after cleaning, and how many were dropped:
- Best model and its MAE, versus the baseline MAE:
- Acropolis premium in one sentence:
- Limitations: asking prices only, one snapshot (no seasonality), host-grouped split.